# 01 · Khám phá dữ liệu: metadata Jamendo và âm thanh số

**Notebook này làm hai việc**
1. **Phần A:** đọc metadata MTG-Jamendo, đếm tag, thử gom mood thành 4 nhóm (valence × arousal) để biết tập con có khả thi không.
2. **Phần B:** hiểu âm thanh dưới dạng số: lấy mẫu → waveform → STFT → mel-spectrogram.

**Cách chạy**
- Cài thư viện: `pip install -r requirements.txt`
- Chỉnh đường dẫn ở ô **Cấu hình** bên dưới.
- Chưa có dữ liệu thật thì notebook tự dùng **dữ liệu giả (DEMO)** để bạn chạy thử và thấy luồng làm việc. Số liệu DEMO **không có ý nghĩa**, chỉ để minh hoạ.

> Nguyên tắc cả dự án: thử nghiệm ở notebook trước; hàm nào ổn định thì chuyển sang `src/*.py`.

In [ ]:
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sps
import librosa
import librosa.display
from IPython.display import display

# ============ CẤU HÌNH — chỉnh ở đây ============
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METADATA_TSV = ROOT / "data" / "autotagging_moodtheme.tsv"  # file metadata MTG-Jamendo (sau khi tải)
SAMPLE_FILE = None   # đặt Path("đường/dẫn/tới/bài.mp3") để thử với 1 bài thật
SR = 22050           # tần số lấy mẫu dùng xuyên suốt dự án (Hz)
# ================================================

plt.rcParams["figure.dpi"] = 110
print("ROOT =", ROOT)

---
## Phần A — Metadata: đếm tag để quyết định tập con

### A1. Định dạng file
Mỗi dòng của file `.tsv` là một bài hát: **5 cột cố định** (`TRACK_ID`, `ARTIST_ID`, `ALBUM_ID`, `PATH`, `DURATION`) rồi đến **một số lượng tag thay đổi**, mỗi tag có dạng `loại---tên`, ví dụ `genre---pop`, `mood/theme---happy`, `instrument---guitar`.

Vì số cột mỗi dòng khác nhau nên **không dùng thẳng `pd.read_csv`**. Hàm bên dưới tự tách 5 cột đầu và gom phần còn lại thành danh sách tag.

> Định dạng này dựa theo mô tả của repo MTG-Jamendo. Nếu file thật khác, hàm `read_jamendo_tsv` là chỗ duy nhất cần sửa. Hãy mở vài dòng đầu của file thật để đối chiếu.

In [ ]:
def read_jamendo_tsv(path):
    # Đọc TSV Jamendo: 5 cột cố định + số tag thay đổi -> DataFrame, cột tags là list.
    rows = []
    with open(path, encoding="utf-8") as f:
        lines = f.read().splitlines()
    if lines and lines[0].startswith("TRACK_ID"):
        lines = lines[1:]  # bỏ dòng tiêu đề
    for line in lines:
        parts = line.split("\t")
        if len(parts) < 5:
            continue
        track_id, artist_id, album_id, rel_path, duration = parts[:5]
        rows.append({
            "track_id": track_id,
            "artist_id": artist_id,
            "album_id": album_id,
            "path": rel_path,
            "duration": float(duration),
            "tags": parts[5:],
        })
    return pd.DataFrame(rows)


def make_demo_tsv(path, n=400, seed=0):
    # Tạo file giả cùng định dạng để chạy thử khi chưa có dữ liệu thật (DEMO).
    rng = np.random.default_rng(seed)
    genres = ["pop", "rock", "electronic", "ambient", "jazz", "classical", "hiphop"]
    quads = {
        "q1": ["happy", "energetic", "upbeat", "fun"],
        "q2": ["dark", "dramatic", "heavy"],
        "q3": ["sad", "melancholic"],
        "q4": ["calm", "relaxing", "soft"],
    }
    others = ["film", "background", "corporate", "nature"]
    lines = ["TRACK_ID\tARTIST_ID\tALBUM_ID\tPATH\tDURATION\tTAGS"]
    for i in range(n):
        q = rng.choice(list(quads))
        moods = list(rng.choice(quads[q], size=rng.integers(1, 3), replace=False))
        if rng.random() < 0.2:  # một phần bài bị lẫn mood của nhóm khác
            other_q = rng.choice([k for k in quads if k != q])
            moods.append(str(rng.choice(quads[other_q])))
        if rng.random() < 0.3:
            moods.append(str(rng.choice(others)))
        tags = [f"genre---{rng.choice(genres)}"] + [f"mood/theme---{m}" for m in moods]
        lines.append("\t".join([f"{i}", f"{i % 40}", f"{i % 90}", f"{i % 100:02d}/{i}.mp3", "30.0"] + tags))
    Path(path).write_text("\n".join(lines), encoding="utf-8")


if METADATA_TSV.exists():
    DEMO_META = False
    meta_path = METADATA_TSV
else:
    DEMO_META = True
    meta_path = Path(tempfile.mkdtemp()) / "demo_meta.tsv"
    make_demo_tsv(meta_path)
    print("⚠️  KHÔNG thấy", METADATA_TSV)
    print("⚠️  Đang dùng metadata GIẢ (DEMO). Số liệu bên dưới chỉ để minh hoạ luồng làm việc.")

df = read_jamendo_tsv(meta_path)
print(f"Số bài: {len(df)} | thời lượng trung bình: {df['duration'].mean():.1f} s")
df.head()

### A2. Tách tag thành bảng (một dòng = một cặp bài–tag)
`explode` biến danh sách tag của mỗi bài thành nhiều dòng; sau đó tách `loại---tên`. Từ bảng này đếm theo loại (genre / mood-theme / instrument) rất dễ.

In [ ]:
tag_rows = df[["track_id", "tags"]].explode("tags").dropna()
tag_rows[["category", "tag"]] = tag_rows["tags"].str.split("---", n=1, expand=True)
tag_rows = tag_rows.dropna(subset=["tag"])

print("Số lượt gắn tag theo loại:")
print(tag_rows["category"].value_counts(), "\n")

for cat in ["genre", "mood/theme"]:
    sub = tag_rows[tag_rows["category"] == cat]
    print(f"--- Top tag của '{cat}' (số bài) ---")
    print(sub["tag"].value_counts().head(15), "\n")

### A3. Gom mood thành 4 nhóm theo valence × arousal
Mood theo mô hình hai trục:
- **Valence**: tích cực ↔ tiêu cực.
- **Arousal**: năng lượng cao ↔ thấp.

Cho ra 4 nhóm: *vui–năng lượng cao*, *căng/tối–năng lượng cao*, *buồn–năng lượng thấp*, *bình yên–năng lượng thấp*. Gom vậy có hai lợi ích: ít lớp hơn (dễ học hơn), và có căn cứ lý thuyết để giải thích khi vấn đáp.

> ⚠️ Từ điển `MOOD_QUADRANTS` bên dưới là **bản nháp**: tên tag là gợi ý, chưa đối chiếu với danh sách chính thức. Ô code sẽ **tự cảnh báo tag nào không có trong dữ liệu** để bạn sửa. Việc xếp tag nào vào nhóm nào cũng là quyết định thiết kế, cần giải thích được.

**Quy tắc lọc:** chỉ giữ bài có tag mood rơi vào **đúng một nhóm**. Bài lẫn nhiều nhóm có nhãn mơ hồ, sẽ làm nhiễu mô hình.

In [ ]:
MOOD_QUADRANTS = {
    "vui_nang_luong_cao":       ["happy", "energetic", "upbeat", "fun", "party", "uplifting"],
    "cang_toi_nang_luong_cao":  ["dark", "dramatic", "heavy", "powerful"],
    "buon_nang_luong_thap":     ["sad", "melancholic", "emotional"],
    "binh_yen_nang_luong_thap": ["calm", "relaxing", "meditative", "soft", "mellow", "dream"],
}
tag2quad = {t: q for q, tags in MOOD_QUADRANTS.items() for t in tags}

mood_rows = tag_rows[tag_rows["category"] == "mood/theme"].copy()
mood_tags_in_data = set(mood_rows["tag"])

# 1) Kiểm tra từ điển với dữ liệu thật
missing = sorted(set(tag2quad) - mood_tags_in_data)
unmapped = sorted(mood_tags_in_data - set(tag2quad))
if missing:
    print("⚠️ Tag có trong từ điển nhưng KHÔNG có trong dữ liệu (kiểm tra tên):", missing)
print("Tag mood có trong dữ liệu nhưng chưa xếp nhóm:", unmapped, "\n")

# 2) Mỗi bài thuộc những nhóm nào?
mood_rows["quadrant"] = mood_rows["tag"].map(tag2quad)
quads_per_track = (mood_rows.dropna(subset=["quadrant"])
                   .groupby("track_id")["quadrant"].agg(set))
n_quads = quads_per_track.map(len)
print("Số nhóm mood mà mỗi bài rơi vào (1 là sạch):")
print(n_quads.value_counts().sort_index(), "\n")

# 3) Giữ bài thuộc đúng 1 nhóm
clean = quads_per_track[n_quads == 1].map(lambda s: next(iter(s)))
print(f"Còn lại {len(clean)}/{len(df)} bài sau khi lọc. Số bài mỗi nhóm:")
print(clean.value_counts())

### A4. Mỗi nhóm mood có đủ thể loại đa dạng không?
Bảng chéo *nhóm mood × thể loại* cho thấy dữ liệu có **cân bằng** không. Nếu một nhóm mood gần như chỉ có một thể loại, mô hình có thể học "thể loại" thay vì "mood" rồi nhầm.

In [ ]:
genre_rows = tag_rows[tag_rows["category"] == "genre"]
top_genres = genre_rows["tag"].value_counts().head(8).index

g = genre_rows[genre_rows["track_id"].isin(clean.index) & genre_rows["tag"].isin(top_genres)].copy()
g["quadrant"] = g["track_id"].map(clean)
ct = pd.crosstab(g["quadrant"], g["tag"])
display(ct)

ax = ct.plot(kind="bar", stacked=True, figsize=(8, 3.5))
ax.set_ylabel("số bài")
ax.set_title("Nhóm mood × thể loại (top 8 thể loại)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### A5. Cách đọc các con số để ra quyết định
- Mỗi nhóm mood nên có **vài trăm bài trở lên** thì mới chia train/val/test được. Nhóm quá ít → gộp nhóm, hoặc chấp nhận bỏ nhóm đó.
- Lệch giữa các nhóm (vd 2000 vs 150) → **lấy mẫu cân bằng** (mỗi nhóm cùng một số bài) hoặc dùng macro-F1 khi đánh giá.
- Nếu bảng chéo cho thấy một nhóm gần như một thể loại → ghi nhận như một hạn chế của dữ liệu.

**Đây là phần Đăng nên tự chạy trên dữ liệu thật** để hiểu data trước khi tải audio.

---
## Phần B — Âm thanh dưới dạng số

### B1. Lấy mẫu
Âm thanh là sóng liên tục. Máy tính đo biên độ **$f_s$ lần mỗi giây** (tần số lấy mẫu), ra một dãy số $x[0], x[1], \dots$.

- Với $f_s = 22050$ Hz, mỗi giây có 22050 số.
- **Nyquist:** chỉ biểu diễn đúng được tần số tới $f_s/2$ (ở đây 11025 Hz). Cao hơn bị méo (aliasing). Vì vậy cả dự án thống nhất **mono, 22050 Hz** để các bài so sánh được với nhau.

Hàm `load_audio` nạp file thật (nếu có), còn không thì tạo tín hiệu giả: 4 nốt nhạc có họa âm (0–4 s) rồi một tiếng quét tần từ 200 Hz lên 4000 Hz (4–6 s). Tín hiệu giả có cấu trúc rõ nên nhìn spectrogram sẽ hiểu STFT làm gì.

In [ ]:
def make_demo_signal(sr=SR, dur=6.0, seed=0):
    # Tín hiệu giả: 4 nốt (C4 E4 G4 C5) có họa âm + chirp 200 -> 4000 Hz + nhiễu nhỏ.
    rng = np.random.default_rng(seed)
    t = np.arange(int(sr * dur)) / sr
    y = np.zeros_like(t)
    for i, f0 in enumerate([261.63, 329.63, 392.00, 523.25]):
        idx = (t >= i) & (t < i + 1.0)
        tt = t[idx] - i
        tone = sum((1.0 / h) * np.sin(2 * np.pi * f0 * h * tt) for h in range(1, 5))
        y[idx] += np.exp(-3 * tt) * tone          # biên độ giảm dần như nốt đàn
    idx = t >= 4.0
    y[idx] += 0.5 * sps.chirp(t[idx] - 4.0, f0=200, f1=4000, t1=2.0, method="logarithmic")
    y += 0.01 * rng.standard_normal(len(t))
    return (y / np.max(np.abs(y)) * 0.9).astype(np.float32)


def load_audio(path=None, sr=SR):
    # Nạp audio -> mono, resample về sr. Không có file thật thì dùng tín hiệu giả (DEMO).
    if path is not None and Path(path).exists():
        y, _ = librosa.load(path, sr=sr, mono=True)
        return y, "file"
    warnings.warn("Không có file thật -> dùng tín hiệu giả (DEMO).")
    return make_demo_signal(sr), "demo"


y, src = load_audio(SAMPLE_FILE)
print(f"nguồn: {src} | sr = {SR} Hz | số mẫu = {len(y)} | thời lượng = {len(y)/SR:.2f} s")
print(f"dtype = {y.dtype} | biên độ trong [{y.min():.2f}, {y.max():.2f}]")

### B2. Waveform
Waveform là biên độ theo thời gian. Hình dưới có hai khung: toàn bài, và **phóng to 20 ms** để thấy từng **mẫu** (mỗi chấm là một số trong mảng `y`).

Waveform cho biết to/nhỏ theo thời gian, nhưng **khó thấy nốt nào, tần số nào**. Vì vậy cần STFT.

In [ ]:
t = np.arange(len(y)) / SR
a = int(0.5 * SR)
b = a + int(0.020 * SR)   # 20 ms => khoảng 441 mẫu

fig, axes = plt.subplots(2, 1, figsize=(9, 5))
axes[0].plot(t, y, lw=0.6)
axes[0].set_title("Waveform toàn bài")
axes[0].set_xlabel("thời gian (s)")
axes[1].plot(t[a:b], y[a:b], ".-", lw=0.8, ms=3)
axes[1].set_title(f"Phóng to 20 ms (≈ {b - a} mẫu)")
axes[1].set_xlabel("thời gian (s)")
plt.tight_layout()
plt.show()

### B3. STFT: nhìn âm thanh theo (thời gian, tần số)
Cắt tín hiệu thành các **khung** ngắn dài $N$ mẫu (`n_fft`), nhân với cửa sổ $w[n]$, rồi biến đổi Fourier từng khung. Hai khung liên tiếp cách nhau $H$ mẫu (`hop_length`):

$$X(m, k) = \sum_{n=0}^{N-1} x[n + mH]\, w[n]\, e^{-j 2\pi k n / N}$$

Mỗi giá trị $|X(m,k)|$ cho biết **tại khung thứ $m$, có bao nhiêu năng lượng ở tần số thứ $k$**. Kết quả là ma trận cỡ $(1 + N/2) \times T$.

**Đánh đổi quan trọng nhất (nên nắm chắc):**
- Độ phân giải tần số $\Delta f = f_s / N$. $N$ lớn → phân biệt được hai tần số sát nhau.
- Độ dài khung $N / f_s$ giây. $N$ lớn → khung dài → **mờ về thời gian**.
- Không thể tốt cả hai cùng lúc. Với $f_s = 22050$ và $N = 2048$: $\Delta f \approx 10.8$ Hz, khung ≈ 93 ms, nhảy $H = 512$ ≈ 23 ms.

Hình dưới so sánh $N = 512$ và $N = 4096$ trên cùng tín hiệu. Quan sát: $N$ nhỏ thấy rõ **thời điểm** nốt chuyển nhưng các họa âm dính nhau; $N$ lớn tách các họa âm rõ nhưng ranh giới thời gian bị nhòe.

Đơn vị **dB** ($20\log_{10}$ biên độ) dùng vì tai người nghe theo thang log: nếu để thang tuyến tính, hầu như chỉ thấy vài điểm sáng nhất.

In [ ]:
def spec_db(y, n_fft, hop):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop))
    return S, librosa.amplitude_to_db(S, ref=np.max)


fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, n_fft in zip(axes, [512, 4096]):
    hop = n_fft // 4
    S, S_db = spec_db(y, n_fft, hop)
    img = librosa.display.specshow(S_db, sr=SR, hop_length=hop, x_axis="time", y_axis="hz", ax=ax)
    ax.set_ylim(0, 6000)
    ax.set_title(f"n_fft={n_fft}: Δf≈{SR / n_fft:.1f} Hz, khung≈{1000 * n_fft / SR:.0f} ms")
    print(f"n_fft={n_fft:5d} | hop={hop:4d} | S.shape = {S.shape}  (tần số × khung)")
fig.colorbar(img, ax=axes, format="%+2.0f dB", label="dB (so với đỉnh)")
plt.show()

### B4. Mel-spectrogram: nén trục tần số theo cách tai người nghe
Tai người **nhạy với chênh lệch ở tần số thấp hơn nhiều so với tần số cao**: 100 → 200 Hz nghe khác rõ, còn 8000 → 8100 Hz gần như không. Thang **mel** mô phỏng điều này, một công thức phổ biến:

$$m = 2595 \log_{10}\!\left(1 + \frac{f}{700}\right)$$

Cách làm: dùng bộ lọc tam giác dày ở tần số thấp, thưa ở cao (hình dưới), gom 1025 bin tần số của STFT thành chỉ **128 băng mel**. Ma trận nhỏ hơn nhiều, mà vẫn giữ phần thông tin quan trọng với việc nghe. Đây là đầu vào chuẩn của nhiều mô hình âm thanh và là bước đệm để tính MFCC ở notebook sau.

Lưu ý: mel-spectrogram dùng **năng lượng** (bình phương biên độ), rồi đổi sang dB.

In [ ]:
N_FFT, HOP, N_MELS = 2048, 512, 128
S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP))
M = librosa.feature.melspectrogram(S=S ** 2, sr=SR, n_mels=N_MELS)   # dùng năng lượng
M_db = librosa.power_to_db(M, ref=np.max)
print(f"STFT tuyến tính: {S.shape}  ->  mel: {M.shape}  (giảm {S.shape[0] / M.shape[0]:.1f} lần số hàng)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
librosa.display.specshow(librosa.amplitude_to_db(S, ref=np.max), sr=SR, hop_length=HOP,
                         x_axis="time", y_axis="hz", ax=axes[0])
axes[0].set_title("STFT tuyến tính (1025 bin)")
img = librosa.display.specshow(M_db, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[1])
axes[1].set_title(f"Mel-spectrogram ({N_MELS} băng)")
fig.colorbar(img, ax=axes, format="%+2.0f dB")
plt.show()

# Bộ lọc mel: các tam giác dày ở tần số thấp, thưa ở tần số cao
fb = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=20)
freqs = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
plt.figure(figsize=(9, 2.8))
plt.plot(freqs, fb.T, lw=0.9)
plt.xlabel("tần số (Hz)"); plt.title("20 bộ lọc mel đầu (minh hoạ)")
plt.tight_layout(); plt.show()

---
## Tự thử (rất nên làm, vì đây là cách hiểu thật)

1. Đổi `n_fft` thành 256 và 8192 ở B3. Hình đổi thế nào? Nêu được vì sao theo công thức $\Delta f = f_s/N$.
2. Đổi `hop_length` (vd `n_fft // 2` và `n_fft // 8`). Số khung $T$ thay đổi ra sao?
3. Đặt `SAMPLE_FILE` là một bài thật. Nhận ra nhịp trống, nốt dài, tiếng hát trên spectrogram chưa?
4. Đổi `N_MELS` thành 40 và 256. Mel-spectrogram mất và giữ những gì?
5. (Phần A) Thêm/bớt tag trong `MOOD_QUADRANTS`. Số bài mỗi nhóm thay đổi thế nào, nhóm nào bị hụt?

## Câu hỏi tự kiểm tra (kiểu vấn đáp)
- Vì sao phải đưa mọi bài về cùng tần số lấy mẫu và mono?
- STFT cho ta thông tin gì mà waveform không cho? Cái giá phải trả là gì?
- Vì sao dùng thang dB và thang mel?
- Vì sao lọc bỏ bài có mood rơi vào nhiều nhóm?
- Nhìn bảng chéo mood × thể loại, thấy rủi ro gì với mô hình?

**Bước tiếp theo:** `02_genre_similarity.ipynb` — từ mel-spectrogram trích đặc trưng (MFCC, chroma, spectral, tempo), chuẩn hoá, phân loại thể loại, tìm bài giống.